# ILP -> Platform Schedule conversion test

This notebook tests conversion of ILP outputs (`variables_a/z/x`, `stream_dict`) into the platform `Schedule` object.

Notes:
- Time conversion is controlled by `time_scale_to_ns`.
- For rctsn outputs in microseconds, use `time_scale_to_ns = 1000` to map into platform nanosecond fields.

In [ ]:
%matplotlib inline

from pathlib import Path
from collections import Counter

from ilp_to_platform import (
    load_ilp_artifacts,
    convert_ilp_artifacts_to_schedule,
    summarize_conversion,
)


In [ ]:
RUN_DIR = Path("rctsn/data/output/run_20260218_140154_f169c70e")

# Platform uses *_ns fields in schedule objects, so we scale ILP us->ns by default.
time_scale_to_ns = 1000

artifacts = load_ilp_artifacts(RUN_DIR)
result = convert_ilp_artifacts_to_schedule(
    artifacts,
    time_scale_to_ns=time_scale_to_ns,
    duration_mode="robust",
    compact_periodic=False,  # exact per repetition slots
)

summary = summarize_conversion(result)
summary

In [ ]:
print("Cycle (ns):", result.cycle_ns)
print("Active streams:", len(result.active_streams))
print("Selected paths:", len(result.selected_paths))
print("Slots:", len(result.schedule))
print("Ports used:", len(result.port_tuple_to_port_id))
print("Dropped x-entries:", result.dropped_entries)
print("Wrapped entries:", result.wrapped_entries)

print("\nSample selected paths:")
for s in sorted(list(result.selected_paths.keys()))[:10]:
    print(f"stream {s} -> path {result.selected_paths[s]}")

print("\nSample port mapping:")
for p, pid in list(result.port_tuple_to_port_id.items())[:10]:
    print(pid, p)

In [ ]:
conflicts = result.schedule.validate_conflicts()
print("Conflicts:", len(conflicts))
if conflicts:
    print("First 5 conflicts:")
    for c1, c2 in conflicts[:5]:
        print(
            f"port={c1.port_id} s{c1.stream_id} [{c1.start_ns},{c1.end_ns}]",
            "<->",
            f"s{c2.stream_id} [{c2.start_ns},{c2.end_ns}]",
        )

In [ ]:
# Pick busiest ports for readable plotting
counts = Counter([slot.port_id for slot in result.schedule])
top_ports = [pid for pid, _ in counts.most_common(8)]
print("Top ports:", top_ports)

fig, axs = result.schedule.plot(
    tick=1000,            # 1000 ns = 1 us on x-axis
    label=False,
    port_ids=top_ports,
)


In [ ]:
# Optional: compare with compact periodic representation
result_compact = convert_ilp_artifacts_to_schedule(
    artifacts,
    time_scale_to_ns=time_scale_to_ns,
    duration_mode="robust",
    compact_periodic=True,
)

print("Exact slots:", len(result.schedule))
print("Compact slots:", len(result_compact.schedule))
print("Compact conflicts:", len(result_compact.schedule.validate_conflicts()))